# Criação dos rasters alvo para Random Forest

Este notebook cria os rasters alvo dos cenários C2, C4 e C6. Os valores produzidos são `-1` para NoData ou área não elegível, `0` para área elegível não ardida e `1` para área elegível ardida.

In [ ]:
import os
import sys

sys.path.append("/code/scripts")

from rf_target_utils import check_target_blocks, count_to_rf_target_blocks

## Configuração

Alterar apenas `scenario` e `area`.

In [ ]:
# ALTERAR APENAS ESTAS DUAS VARIÁVEIS

scenario = "C6"
area = "extremadura"

# C2 e C4: area = "centro"
# C6: area = "centro" ou "extremadura"


# C2 — Centro | ICNF | 1995–2024
if scenario == "C2":
    lr_scenario = "C1"
    source = "icnf"
    train_period = "1995_2024"


# C4 — Centro | ICNF | 2008–2024
elif scenario == "C4":
    lr_scenario = "C3"
    source = "icnf"
    train_period = "2008_2024"


# C6 — Centro ou Extremadura | EFFIS | 2008–2024
elif scenario == "C6":
    lr_scenario = "C5"
    source = "effis"
    train_period = "2008_2024"


base = f"/code/data/processed/{area}"

count_dir = (
    f"{base}/area_ardida/"
    f"{source}/raster_count"
)

target_dir = (
    f"/code/data/results/{area}/"
    f"{scenario}/lri_model/target"
)

ref_rst = (
    f"/code/data/results/{area}/"
    f"{lr_scenario}/final/res_lri.tif"
)

count_train = (
    f"{count_dir}/"
    f"rst_ba_{train_period}.tif"
)

count_valid = (
    f"{count_dir}/"
    "rst_ba_2025.tif"
)

target_train = (
    f"{target_dir}/"
    f"rst_ba_{train_period}_target.tif"
)

target_valid = (
    f"{target_dir}/"
    "rst_ba_2025_target.tif"
)

os.makedirs(target_dir, exist_ok=True)

print("Cenário RF:", scenario)
print("Área:", area)
print("Cenário LR de referência:", lr_scenario)
print("Fonte das áreas ardidas:", source)
print("Período de treino:", train_period.replace("_", "–"))
print("Máscara burnable:", ref_rst)


## Criação dos rasters alvo

In [ ]:
targets = {
    train_period: (
        count_train,
        target_train,
    ),
    "2025": (
        count_valid,
        target_valid,
    ),
}


for period, paths in targets.items():
    count_rst, target_rst = paths

    print("\n" + "=" * 60)
    print("Criar target:", period.replace("_", "–"))
    print("=" * 60)

    count_to_rf_target_blocks(
        count_rst=count_rst,
        ref_rst=ref_rst,
        out_rst=target_rst,
        nodata=-1,
        block_size=512,
    )


## Verificação dos resultados

In [ ]:
for target_rst in [
    target_train,
    target_valid,
]:
    print("\n" + target_rst)
    check_target_blocks(target_rst)
